In [ ]:
# Prepare Bloomberg input file
#
# Starting inputs are minimal. Bankrupt input has bankruptcy_date, ticker, name,
# bloomberg_identifier. Non-bankrupt input has ticker, name, Sector, Industry.
# This notebook creates all additional fields in the combined output.

In [ ]:
%pip install pandas numpy pathlib


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    
BANKRUPT_PATH = ROOT / "data/raw/bankrupt/data.csv"
NONBANKRUPT_PATH = ROOT / "data/raw/nonbankrupt/data.csv"
OUTPUT_DIR = ROOT / "data/intermediate"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOOKBACK_YEARS = 5
RANDOM_STATE = 42

In [ ]:
bankrupt = pd.read_csv(BANKRUPT_PATH, dtype={"ticker": "string", "name": "string", "bloomberg_identifier": "string"})
required = {"bankruptcy_date", "ticker", "name", "bloomberg_identifier"}
missing = required - set(bankrupt.columns)
assert not missing, f"Missing columns: {sorted(missing)}"
bankrupt["bankruptcy_date"] = pd.to_datetime(bankrupt["bankruptcy_date"], errors="coerce")
bankrupt["ticker"] = bankrupt["ticker"].str.strip().str.upper()
bankrupt["name"] = bankrupt["name"].str.strip()
bankrupt["bloomberg_identifier"] = bankrupt["bloomberg_identifier"].fillna("").str.strip()
bankrupt = bankrupt.loc[bankrupt["bloomberg_identifier"].ne("")].copy()
bankrupt["company_id"] = "SA_" + bankrupt.index.astype(str)
bankrupt["prediction_date"] = bankrupt["bankruptcy_date"] - pd.Timedelta(days=90)
bankrupt["start_date"] = bankrupt["prediction_date"] - pd.DateOffset(years=LOOKBACK_YEARS)
bankrupt["end_date"] = bankrupt["prediction_date"]
bankrupt = bankrupt.sort_values(["bankruptcy_date", "company_id"]).copy()
display(bankrupt)

In [ ]:
bankrupt_output = bankrupt[["company_id", "bankruptcy_date", "ticker", "name", "bloomberg_identifier", "prediction_date", "start_date", "end_date"]].copy()

In [22]:
nonbankrupt = pd.read_csv(NONBANKRUPT_PATH, dtype={"ticker": "string", "name": "string"})
required_nonbankrupt = {"ticker", "name", "Sector", "Industry"}
missing_nonbankrupt = required_nonbankrupt - set(nonbankrupt.columns)
if missing_nonbankrupt:
    raise ValueError(f"Non-bankrupt CSV is missing columns: {sorted(missing_nonbankrupt)}")
nonbankrupt["ticker"] = nonbankrupt["ticker"].str.strip().str.upper()
nonbankrupt["name"] = nonbankrupt["name"].str.strip()
nonbankrupt["bloomberg_identifier"] = nonbankrupt["ticker"] + " US Equity"
nonbankrupt["company_id"] = "NB_" + nonbankrupt.index.astype(str)
# Sample unique usable tickers, retaining the original company IDs.
nonbankrupt = nonbankrupt.loc[nonbankrupt["ticker"].fillna("").ne("")].drop_duplicates("ticker").copy()
for column in ["Sector", "Industry"]:
    nonbankrupt[column] = nonbankrupt[column].astype("string").fillna("Unknown").str.strip().replace("", "Unknown")
nonbankrupt = nonbankrupt.sort_values(["Sector", "Industry", "ticker"])
NONBANKRUPT_N = len(bankrupt)
if NONBANKRUPT_N == 0:
    raise ValueError("No bankrupt records have a nonempty Bloomberg identifier.")
if NONBANKRUPT_N > len(nonbankrupt):
    raise ValueError(f"Need {NONBANKRUPT_N} non-bankrupt companies, but only {len(nonbankrupt)} unique usable tickers are available.")

# Cover every sector-industry group, then approach pool proportions.
industry_sampling = nonbankrupt.groupby(["Sector", "Industry"], sort=True).size().rename("available").to_frame()
if NONBANKRUPT_N < len(industry_sampling):
    raise ValueError(f"A sample of {NONBANKRUPT_N} cannot cover all {len(industry_sampling)} sector-industry groups.")
industry_sampling["target"] = NONBANKRUPT_N * industry_sampling["available"] / len(nonbankrupt)
industry_sampling["selected"] = 1
for _ in range(NONBANKRUPT_N - len(industry_sampling)):
    eligible = industry_sampling["selected"] < industry_sampling["available"]
    deficit = (industry_sampling["target"] - industry_sampling["selected"]).loc[eligible]
    group_key = deficit.idxmax()
    industry_sampling.at[group_key, "selected"] += 1

nonbankrupt = pd.concat([
    group.sample(n=int(industry_sampling.at[group_key, "selected"]), random_state=RANDOM_STATE)
    for group_key, group in nonbankrupt.groupby(["Sector", "Industry"], sort=True)
]).sort_values(["Sector", "Industry", "ticker"]).copy()
sector_sampling = industry_sampling.groupby(level="Sector")[["available", "selected"]].sum()
sector_sampling["pool_pct"] = 100 * sector_sampling["available"] / industry_sampling["available"].sum()
sector_sampling["sample_pct"] = 100 * sector_sampling["selected"] / NONBANKRUPT_N
display(sector_sampling.round(2))
print(f"Selected {len(nonbankrupt)} non-bankrupt companies to match {len(bankrupt)} bankrupt records; covered all {len(sector_sampling)} sectors and {len(industry_sampling)} sector-industry groups.")
nonbankrupt["prediction_date"] = pd.Timestamp("2021-06-30")
nonbankrupt["start_date"] = nonbankrupt["prediction_date"] - pd.DateOffset(years=LOOKBACK_YEARS)
nonbankrupt["end_date"] = nonbankrupt["prediction_date"]
nonbankrupt_output = nonbankrupt[["company_id", "ticker", "name", "Sector", "Industry", "bloomberg_identifier", "prediction_date", "start_date", "end_date"]].copy()
nonbankrupt_output.to_csv(OUTPUT_DIR / "nonbankrupt_bloomberg_input.csv", index=False)
display(nonbankrupt_output.head())
print("Saved:", OUTPUT_DIR / "nonbankrupt_bloomberg_input.csv")


,available,selected,pool_pct,sample_pct
Sector,,,,
Basic Materials,46,6,1.40,2.23
Consumer Discretionary,740,63,22.45,23.42
Consumer Staples,98,9,2.97,3.35
Energy,138,13,4.19,4.83
Health Care,825,61,25.03,22.68
Industrials,460,48,13.96,17.84
Real Estate,248,20,7.52,7.43
Technology,531,35,16.11,13.01
Telecommunications,61,4,1.85,1.49


Selected 269 non-bankrupt companies to match 269 bankrupt records; covered all 10 sectors and 173 sector-industry groups.


,company_id,ticker,name,Sector,Industry,bloomberg_identifier,prediction_date,start_date,end_date
1092,NB_1092,FBIN,Fortune Brands Innovations Inc. Common Stock,Basic Materials,Forest Products,FBIN US Equity,2021-06-30,2016-06-30,2021-06-30
2950,NB_2950,TROX,Tronox Holdings plc Ordinary Shares (UK),Basic Materials,Major Chemicals,TROX US Equity,2021-06-30,2016-06-30,2021-06-30
3043,NB_3043,USAU,U.S. Gold Corp. Common Stock,Basic Materials,Metal Mining,USAU US Equity,2021-06-30,2016-06-30,2021-06-30
1533,NB_1533,IPX,IperionX Limited American Depositary Share,Basic Materials,Other Metals and Minerals,IPX US Equity,2021-06-30,2016-06-30,2021-06-30
658,NB_658,CLW,Clearwater Paper Corporation Common Stock,Basic Materials,Paper,CLW US Equity,2021-06-30,2016-06-30,2021-06-30


Saved: C:\Users\MyPC\myprojects\bankruptcy-prediction\data\intermediate\nonbankrupt_bloomberg_input.csv


In [ ]:
# Combine the matched-size samples
#
# Include every bankrupt record with a nonempty Bloomberg identifier.
# The non-bankrupt sample covers every sector-industry group; minimum
# coverage overrepresents small groups relative to the candidate pool.

In [26]:
bankrupt_list = bankrupt_output[["company_id", "bankruptcy_date", "ticker", "name", "bloomberg_identifier", "prediction_date", "start_date", "end_date"]].copy()
bankrupt_list["company_type"] = "bankrupt"
nonbankrupt_list = nonbankrupt_output[["company_id", "ticker", "name", "Sector", "Industry", "bloomberg_identifier", "prediction_date", "start_date", "end_date"]].copy()
nonbankrupt_list["company_type"] = "nonbankrupt"
bloomberg_input = pd.concat([bankrupt_list, nonbankrupt_list], ignore_index=True)
bloomberg_input = bloomberg_input[["company_id", "company_type", "bankruptcy_date", "ticker", "name", "Sector", "Industry", "bloomberg_identifier", "prediction_date", "start_date", "end_date"]]
bloomberg_input.to_csv(OUTPUT_DIR / "bloomberg_security_list.csv", index=False)
display(bloomberg_input.head(20))
print("Saved:", OUTPUT_DIR / "bloomberg_security_list.csv")


,company_id,company_type,bankruptcy_date,ticker,name,Sector,Industry,bloomberg_identifier,prediction_date,start_date,end_date
0,SA_222,bankrupt,1998-09-16,SNTVQ,"Sun Television and Appliances, Inc.",<NA>,<NA>,SNTVQ US Equity,1998-06-18,1993-06-18,1998-06-18
1,SA_187,bankrupt,1998-10-05,BOST,"Boston Chicken, Inc.",<NA>,<NA>,BCHXQ US Equity,1998-07-07,1993-07-07,1998-07-07
2,SA_135,bankrupt,1998-12-27,JSIBQ,"JumboSports, Inc.",<NA>,<NA>,JSIBQ US Equity,1998-09-28,1993-09-28,1998-09-28
3,SA_4,bankrupt,1998-12-29,SLNT,Salant Corp. (1998),<NA>,<NA>,2777116Q US Equity,1998-09-30,1993-09-30,1998-09-30
4,SA_26,bankrupt,1999-02-11,WIRL,"Wireless One, Inc.",<NA>,<NA>,WIRLQ US Equity,1998-11-13,1993-11-13,1998-11-13
5,SA_172,bankrupt,1999-03-01,PNFTQ,Penn Traffic Co (1999),<NA>,<NA>,PNFTQ US Equity,1998-12-01,1993-12-01,1998-12-01
6,SA_80,bankrupt,1999-03-15,SME,"Service Merchandise Company, Inc.",<NA>,<NA>,SVCDQ US Equity,1998-12-15,1993-12-15,1998-12-15
7,SA_134,bankrupt,1999-09-03,COSE,"Costilla Energy, Inc.",<NA>,<NA>,COSEQ US Equity,1999-06-05,1994-06-05,1999-06-05
8,SA_52,bankrupt,1999-09-16,TRSM,"Trism, Inc.",<NA>,<NA>,2471671D US Equity,1999-06-18,1994-06-18,1999-06-18
9,SA_145,bankrupt,1999-11-04,FEET,"Just For Feet, Inc.",<NA>,<NA>,FEETQ US Equity,1999-08-06,1994-08-06,1999-08-06


Saved: C:\Users\MyPC\myprojects\bankruptcy-prediction\data\intermediate\bloomberg_security_list.csv


In [27]:
# Generate prediction and Bloomberg date windows
#
# Bankrupt-company prediction dates are 90 days before filing. Non-bankrupt
# companies use the fixed prediction date set above. Both groups retain
# five years of history; extra years are excluded during data cleaning.

In [28]:
assert bankrupt_output["company_id"].is_unique
assert nonbankrupt_output["company_id"].is_unique
assert bankrupt_output["bloomberg_identifier"].notna().all()
assert bankrupt_output["bloomberg_identifier"].str.strip().ne("").all()
assert nonbankrupt_output["bloomberg_identifier"].str.endswith(" US Equity").all()
assert nonbankrupt_output["ticker"].is_unique
assert len(nonbankrupt_output) == len(bankrupt_output)
sample_counts = nonbankrupt_output.groupby(["Sector", "Industry"]).size()
assert sample_counts.reindex(industry_sampling.index, fill_value=0).eq(industry_sampling["selected"]).all()
assert len(bloomberg_input) == len(bankrupt_output) + len(nonbankrupt_output)
print("Combined rows:", len(bloomberg_input))
print("Bankrupt rows with blank Bloomberg identifier:", (bankrupt_output["bloomberg_identifier"] == "").sum())

Combined rows: 538
Bankrupt rows with blank Bloomberg identifier: 0
